<a href="https://colab.research.google.com/github/lucasvazelle/nexialog_challlenge/blob/main/version_finale_PEAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# chemin
file_path = "/content/drive/MyDrive/nexialog/data_preprocessed_FINAL.parquet"

# Lire le fichier Parquet
df = pd.read_parquet(file_path)


In [ ]:
# Afficher toutes les colonnes, quelle que soit leur nombre
pd.set_option('display.max_columns', None)

In [ ]:
df.head(5)

,date_hour,code_departement,olt_model,olt_name,peag_nro,boucle,dsp,pop_dns,nb_test_dns,avg_dns_time,std_dns_time,nb_test_scoring,avg_latence_scoring,std_latence_scoring,avg_score_scoring,std_score_scoring,nb_client_total,moment_journee,depts_egaux,meme_region,missing_avg_dns_time,missing_std_dns_time,missing_avg_latence_scoring,missing_std_latence_scoring,missing_avg_score_scoring,missing_std_score_scoring
0,2024-12-01,01,M11,01_olt_3,01_peag_3,BU1464,dsp_1,69_lyon,23,6.845641,0.377309,9,11.437500,0.281250,3.258823,0.375156,32,nuit,False,True,0,0,0,0,0,0
1,2024-12-01,64,M22,64_olt_3774,64_peag_1885,BU112,dsp_25,33_bdx,17,3.583456,0.252478,4,NaN,NaN,NaN,NaN,21,nuit,False,True,0,0,1,1,1,1
2,2024-12-01,64,M22,64_olt_3773,64_peag_1884,BU124,dsp_25,33_bdx,10,3.638700,0.616376,10,9.112500,0.237171,3.465882,0.005581,20,nuit,False,True,0,0,0,0,0,0
3,2024-12-01,64,M22,64_olt_3766,64_peag_1877,BU124,dsp_25,33_bdx,14,4.508839,0.534071,5,9.675000,0.251558,3.465882,0.003946,19,nuit,False,True,0,0,0,0,0,0
4,2024-12-01,64,M22,64_olt_3765,64_peag_1876,BU136,dsp_25,33_bdx,20,4.808550,0.810622,12,9.796875,0.289647,3.404412,0.207982,32,nuit,False,True,0,0,0,0,0,0


In [ ]:
df.columns

Index(['date_hour', 'code_departement', 'olt_model', 'olt_name', 'peag_nro',
       'boucle', 'dsp', 'pop_dns', 'nb_test_dns', 'avg_dns_time',
       'std_dns_time', 'nb_test_scoring', 'avg_latence_scoring',
       'std_latence_scoring', 'avg_score_scoring', 'std_score_scoring',
       'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region',
       'missing_avg_dns_time', 'missing_std_dns_time',
       'missing_avg_latence_scoring', 'missing_std_latence_scoring',
       'missing_avg_score_scoring', 'missing_std_score_scoring'],
      dtype='object')

In [ ]:
# Création des nouvelles variables
df["type_boucle"] = np.where(
    df["boucle"].str.lower().str.startswith("ftth"),
    "FTTH",
    np.where(df["boucle"].str.startswith("BU"), "BU", "Autre")
)

df["type_olt_model"] = np.where(
    df["olt_model"].str.contains("old", case=False, na=False),
    "old",
    "New"
)

In [ ]:
# Répartition des valeurs de type_boucle
repartition_type_boucle = df["type_boucle"].value_counts(dropna=False)  # dropna=False pour inclure les NaN
print("Répartition de type_boucle :")
print(repartition_type_boucle)

# Répartition des valeurs de type_olt_model
repartition_type_olt_model = df["type_olt_model"].value_counts(dropna=False)  # dropna=False pour inclure les NaN
print("\nRépartition de type_olt_model :")
print(repartition_type_olt_model)

Répartition de type_boucle :
type_boucle
BU      2653978
FTTH    1237029
Name: count, dtype: int64

Répartition de type_olt_model :
type_olt_model
New    3390244
old     500763
Name: count, dtype: int64


In [ ]:
# Compter les olt_name avec type_boucle unique ou multiple
stats_type_boucle = df.groupby("olt_name")["type_boucle"].nunique().value_counts()
print("Répartition du nombre de type_boucle uniques par olt_name :")
print(stats_type_boucle)

# Compter les olt_name avec type_olt_model unique ou multiple
stats_type_olt_model = df.groupby("olt_name")["type_olt_model"].nunique().value_counts()
print("\nRépartition du nombre de type_olt_model uniques par olt_name :")
print(stats_type_olt_model)

Répartition du nombre de type_boucle uniques par olt_name :
type_boucle
1    5239
2      57
Name: count, dtype: int64

Répartition du nombre de type_olt_model uniques par olt_name :
type_olt_model
1    5296
Name: count, dtype: int64


In [ ]:
# Convertir date_hour en datetime (avec heures arrondies)
df["date_hour"] = pd.to_datetime(df["date_hour"])

# 1. Encodage cyclique de l'heure (0-23)
df["encodage_heure_sin"] = np.sin(2 * np.pi * df["date_hour"].dt.hour / 24)
df["encodage_heure_cos"] = np.cos(2 * np.pi * df["date_hour"].dt.hour / 24)

# 2. Encodage cyclique du jour de la semaine (0=lundi, 6=dimanche)
df["encodage_jour_semaine_sin"] = np.sin(2 * np.pi * df["date_hour"].dt.weekday / 7)
df["encodage_jour_semaine_cos"] = np.cos(2 * np.pi * df["date_hour"].dt.weekday / 7)

# 3. Week-end (samedi/dimanche)
df["week_end"] = df["date_hour"].dt.weekday >= 5  # 5=samedi, 6=dimanche

IDEES FINALES DE TRAITEMENT PEAG

In [ ]:
# Fonction pour la moyenne pondérée
def weighted_mean(data, weights):
    return np.average(data, weights=weights) if weights.sum() > 0 else np.nan


df_aggregated_peag = df.groupby(['peag_nro', 'date_hour']).agg(
    code_departement=('code_departement', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    olt_model=('olt_model', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    olt_name=('olt_name', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    boucle=('boucle', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    dsp=('dsp', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    pop_dns=('pop_dns', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    nb_test_dns=('nb_test_dns', 'sum'),
    avg_dns_time=('avg_dns_time', lambda x: weighted_mean(x, df.loc[x.index, 'nb_test_dns'])),
    std_dns_time=('std_dns_time', 'mean'),
    nb_test_scoring=('nb_test_scoring', 'sum'),
    avg_latence_scoring=('avg_latence_scoring', lambda x: weighted_mean(x, df.loc[x.index, 'nb_test_scoring'])),
    std_latence_scoring=('std_latence_scoring', 'mean'),
    avg_score_scoring=('avg_score_scoring', lambda x: weighted_mean(x, df.loc[x.index, 'nb_test_scoring'])),
    std_score_scoring=('std_score_scoring', 'mean'),
    nb_client_total=('nb_client_total', 'sum'),
    moment_journee=('moment_journee', 'first'),
    depts_egaux=('depts_egaux', 'sum'),
    meme_region=('meme_region', 'sum'),
    missing_avg_dns_time=('missing_avg_dns_time', 'sum'),
    missing_std_dns_time=('missing_std_dns_time', 'sum'),
    missing_avg_latence_scoring=('missing_avg_latence_scoring', 'sum'),
    missing_std_latence_scoring=('missing_std_latence_scoring', 'sum'),
    missing_avg_score_scoring=('missing_avg_score_scoring', 'sum'),
    missing_std_score_scoring=('missing_std_score_scoring', 'sum'),
    type_boucle=('type_boucle', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    type_olt_model=('type_olt_model', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    encodage_heure_sin=('encodage_heure_sin', 'first'),
    encodage_heure_cos=('encodage_heure_cos', 'first'),
    encodage_jour_semaine_sin=('encodage_jour_semaine_sin', 'first'),
    encodage_jour_semaine_cos=('encodage_jour_semaine_cos', 'first'),
    week_end=('week_end', 'first')
).reset_index()


In [ ]:
output_path = "/content/drive/MyDrive/nexialog/data_aggrege_PEAG.csv"


# Exporter le DataFrame au format CSV
df_aggregated_peag.to_csv(output_path, index=False, encoding='utf-8')

print(f"Fichier exporté avec succès à : {output_path}")

Fichier exporté avec succès à : /content/drive/MyDrive/nexialog/data_aggrege_PEAG.csv


In [ ]:
output_parquet_path = "/content/drive/MyDrive/nexialog/data_aggrege_PEAG.parquet"

# Exporter au format Parquet
df_aggregated_peag.to_parquet(output_parquet_path, index=False, engine='pyarrow')
print(f"Fichier Parquet exporté avec succès à : {output_parquet_path}")

Fichier Parquet exporté avec succès à : /content/drive/MyDrive/nexialog/data_aggrege_PEAG.parquet


In [ ]:
df_aggregated_peag.head(10)

,peag_nro,date_hour,code_departement,olt_model,olt_name,boucle,dsp,pop_dns,nb_test_dns,avg_dns_time,std_dns_time,nb_test_scoring,avg_latence_scoring,std_latence_scoring,avg_score_scoring,std_score_scoring,nb_client_total,moment_journee,depts_egaux,meme_region,missing_avg_dns_time,missing_std_dns_time,missing_avg_latence_scoring,missing_std_latence_scoring,missing_avg_score_scoring,missing_std_score_scoring,type_boucle,type_olt_model,encodage_heure_sin,encodage_heure_cos,encodage_jour_semaine_sin,encodage_jour_semaine_cos,week_end
0,01_peag_1,2024-12-01 00:00:00,01,old0,01_olt_1,BU966,dsp_1,69_lyon,61,4.888439,0.834768,25,10.462500,0.843750,2.852823,0.550380,86,nuit,0,1,0,0,0,0,0,0,BU,old,0.000000,1.000000e+00,-0.781831,0.62349,True
1,01_peag_1,2024-12-01 01:00:00,01,old0,01_olt_1,BU966,dsp_1,69_lyon,58,5.293034,0.986387,24,10.078125,0.328271,3.091176,0.446331,82,nuit,0,1,0,0,0,0,0,0,BU,old,0.258819,9.659258e-01,-0.781831,0.62349,True
2,01_peag_1,2024-12-01 02:00:00,01,old0,01_olt_1,BU966,dsp_1,69_lyon,67,5.103739,0.979593,21,10.312500,0.410792,2.796639,0.513174,88,nuit,0,1,0,0,0,0,0,0,BU,old,0.500000,8.660254e-01,-0.781831,0.62349,True
3,01_peag_1,2024-12-01 03:00:00,01,old0,01_olt_1,BU966,dsp_1,69_lyon,40,5.342719,1.178485,23,10.565217,1.044887,3.013811,0.465058,63,nuit,0,1,0,0,0,0,0,0,BU,old,0.707107,7.071068e-01,-0.781831,0.62349,True
4,01_peag_1,2024-12-01 04:00:00,01,old0,01_olt_1,BU966,dsp_1,69_lyon,50,4.997415,0.856440,26,10.406250,0.397748,2.878846,0.708936,76,nuit,0,1,0,0,0,0,0,0,BU,old,0.866025,5.000000e-01,-0.781831,0.62349,True
5,01_peag_1,2024-12-01 05:00:00,01,old0,01_olt_1,BU966,dsp_1,69_lyon,82,5.068674,1.083690,34,10.439338,0.539730,2.940571,0.588448,116,nuit,0,1,0,0,0,0,0,0,BU,old,0.965926,2.588190e-01,-0.781831,0.62349,True
6,01_peag_1,2024-12-01 06:00:00,01,old0,01_olt_1,BU966,dsp_1,69_lyon,71,5.266225,1.654289,33,10.244318,0.337429,3.143315,0.511603,104,matin,0,1,0,0,0,0,0,0,BU,old,1.000000,6.123234e-17,-0.781831,0.62349,True
7,01_peag_1,2024-12-01 07:00:00,01,old0,01_olt_1,BU966,dsp_1,69_lyon,73,5.323459,1.011645,28,10.125000,0.342327,3.071534,0.494102,101,matin,0,1,0,0,0,0,0,0,BU,old,0.965926,-2.588190e-01,-0.781831,0.62349,True
8,01_peag_1,2024-12-01 08:00:00,01,old0,01_olt_1,BU966,dsp_1,69_lyon,74,5.313649,1.067893,30,10.350000,0.349569,3.050882,0.540183,104,matin,0,1,0,0,0,0,0,0,BU,old,0.866025,-5.000000e-01,-0.781831,0.62349,True
9,01_peag_1,2024-12-01 09:00:00,01,old0,01_olt_1,BU966,dsp_1,69_lyon,52,5.155255,0.846095,17,10.323529,0.484745,2.941868,0.716569,69,matin,0,1,0,0,0,0,0,0,BU,old,0.707107,-7.071068e-01,-0.781831,0.62349,True
